<a href="https://colab.research.google.com/github/Patrick0shea/xAI/blob/Main/milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Qwen3-1.7B on GSM8K Maths Problems

This notebook runs the Qwen3-1.7B model on a subset of the GSM8K dataset to evaluate its performance on maths problems, both with and without its 'thinking' mode enabled. All outputs will be saved for detailed analysis.

### 1. Setup: Install Libraries, Load Model and Tokenizer

In [4]:
# Install/upgrade necessary libraries
%pip install -U transformers accelerate datasets

# Restart kernel after installation to ensure new packages are loaded
# This line caused a TypeError; libraries are installed, proceed without kernel restart for now.
# If issues arise, manually restart the runtime via 'Runtime' -> 'Restart runtime'.
# import os
# os.kill(os.getpid())


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
import re
import json
from tqdm.notebook import tqdm
from datasets import load_dataset

# Set a fixed random seed for reproducibility
# This should be set early to ensure consistent results across runs.
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

# Define paths for saving outputs
OUTPUT_DIR = "outputs"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "qwen3_1.7b_gsm8k.jsonl")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Random seed set to: {RANDOM_SEED}")
print(f"Outputs will be saved to: {OUTPUT_FILE}")

Random seed set to: 42
Outputs will be saved to: outputs/qwen3_1.7b_gsm8k.jsonl


In [3]:
# Load the model
model_id = "Qwen/Qwen3-1.7B"

# T4 GPU does not support bfloat16, so use torch.float16
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Load the tokenizer with padding_side="left"
tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side="left")

print(f"Model '{model_id}' loaded successfully on device: {model.device}")
print(f"Tokenizer '{model_id}' loaded with padding_side: {tokenizer.padding_side}")
print("Model and Tokenizer are ready.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

Model 'Qwen/Qwen3-1.7B' loaded successfully on device: cuda:0
Tokenizer 'Qwen/Qwen3-1.7B' loaded with padding_side: left
Model and Tokenizer are ready.


### 2. Data: Load GSM8K Test Set

In [5]:
# Load the GSM8K test set
print("Loading GSM8K dataset...")
dataset = load_dataset("openai/gsm8k", "main", split="test")
print(f"Dataset loaded with {len(dataset)} examples.")

# Take the first 20 problems
problems = dataset.select(range(20))
print(f"Selected {len(problems)} problems for evaluation.")

# Function to extract the gold answer
def extract_gold_answer(answer_string):
    # The gold answer is the number after "####" in the answer field
    match = re.search(r"#### (\d+(\.\d+)?)", answer_string)
    if match:
        return match.group(1)
    # Fallback: if '####' is not found, try to find the last number
    numbers = re.findall(r'\d+\.\d+|\d+', answer_string)
    if numbers:
        return numbers[-1]
    return None

# Add gold_answer to each problem
problems = problems.map(lambda x: {"gold_answer": extract_gold_answer(x["answer"])})

print("First 3 problems with extracted gold answers:")
for i, problem in enumerate(problems.select(range(min(3, len(problems))))):
    print(f"Problem {i+1}:")
    print(f"  Question: {problem['question']}")
    print(f"  Gold Answer: {problem['gold_answer']}")


Loading GSM8K dataset...


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

...

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Dataset loaded with 1319 examples.
Selected 20 problems for evaluation.


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

First 3 problems with extracted gold answers:
Problem 1:
  Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
  Gold Answer: 18
Problem 2:
  Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
  Gold Answer: 3
Problem 3:
  Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
  Gold Answer: 70000


### 2. Data: Load GSM8K Test Set

In [12]:
# Load the GSM8K test set
print("Loading GSM8K dataset...")
dataset = load_dataset("openai/gsm8k", "main", split="test")
print(f"Dataset loaded with {len(dataset)} examples.")

# Take the first 20 problems
problems = dataset.select(range(20))
print(f"Selected {len(problems)} problems for evaluation.")

# Function to extract the gold answer
def extract_gold_answer(answer_string):
    # The gold answer is the number after "####" in the answer field
    match = re.search(r"#### (\d+(\.\d+)?)", answer_string)
    if match:
        return match.group(1)
    # Fallback: if '####' is not found, try to find the last number
    numbers = re.findall(r'\d+\.\d+|\d+', answer_string)
    if numbers:
        return numbers[-1]
    return None

# Add gold_answer and a unique 'id' to each problem
# The 'id' is generated from the index 'i' for unique identification of each problem.
problems = problems.map(lambda x, i: {"gold_answer": extract_gold_answer(x["answer"]), "id": str(i)}, with_indices=True)

print("First 3 problems with extracted gold answers (with new 'id' field):")
for i, problem in enumerate(problems.select(range(min(3, len(problems))))):
    print(f"Problem {i+1}:")
    print(f"  ID: {problem['id']}")
    print(f"  Question: {problem['question']}")
    print(f"  Gold Answer: {problem['gold_answer']}")

Loading GSM8K dataset...
Dataset loaded with 1319 examples.
Selected 20 problems for evaluation.


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

First 3 problems with extracted gold answers (with new 'id' field):
Problem 1:
  ID: 0
  Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
  Gold Answer: 18
Problem 2:
  ID: 1
  Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
  Gold Answer: 3
Problem 3:
  ID: 2
  Question: Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?
  Gold Answer: 70000


### 3. Prompt Construction

In [7]:
# Helper function to create prompt messages
def create_messages(question, enable_thinking):
    user_message = (
        f"{question}\nSolve this step by step. End your response with 'Final answer: <number>'."
    )
    messages = [
        {"role": "user", "content": user_message}
    ]
    return messages

# Helper function to prepare the prompt for the model
def prepare_prompt(tokenizer, question, enable_thinking):
    messages = create_messages(question, enable_thinking)
    # The enable_thinking argument is passed directly to apply_chat_template
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )
    return prompt

print("Prompt preparation functions defined.")

# Example of a thinking prompt
example_question = problems[0]["question"]
print("\n--- Example Prompt (Thinking ON) ---")
print(prepare_prompt(tokenizer, example_question, enable_thinking=True))

print("\n--- Example Prompt (Thinking OFF) ---")
print(prepare_prompt(tokenizer, example_question, enable_thinking=False))


Prompt preparation functions defined.

--- Example Prompt (Thinking ON) ---
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Solve this step by step. End your response with 'Final answer: <number>'.<|im_end|>
<|im_start|>assistant


--- Example Prompt (Thinking OFF) ---
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Solve this step by step. End your response with 'Final answer: <number>'.<|im_end|>
<|im_start|>assistant
<think>

</think>




### 4. Generation & 5. Parsing

In [13]:
# Generation parameters for thinking mode ON
GENERATION_CONFIG_THINKING_ON = {
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "max_new_tokens": 2048,
}

# Generation parameters for thinking mode OFF
GENERATION_CONFIG_THINKING_OFF = {
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "max_new_tokens": 1024,
}

BATCH_SIZE = 5

# List to store all results temporarily (actual saving happens per batch)
all_results = []

# Function to parse model output
def parse_model_output(generated_tokens_list, tokenizer):
    # Split the generated tokens at the '</think>' token (id 151668)
    # Qwen tokenizer has 151668 as the token ID for </think>
    thinking_token_id = 151668

    if thinking_token_id in generated_tokens_list:
        thinking_tokens = generated_tokens_list[:generated_tokens_list.index(thinking_token_id)]
        response_tokens = generated_tokens_list[generated_tokens_list.index(thinking_token_id) + 1:]
        thinking_text = tokenizer.decode(thinking_tokens, skip_special_tokens=True).strip()
        response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
    else:
        thinking_text = ""
        response_text = tokenizer.decode(generated_tokens_list, skip_special_tokens=True).strip()

    # Extract the model's answer: the number after "Final answer:" or the last number
    model_answer = None
    final_answer_match = re.search(r"Final answer:\s*([\d,\.$]+)", response_text, re.IGNORECASE)
    if final_answer_match:
        model_answer = final_answer_match.group(1).replace(",", "").replace("$", "")
    else:
        # Fallback: if 'Final answer:' is not found, try to find the last number in the response
        # This regex looks for floating point numbers or integers
        numbers = re.findall(r'\d+\.\d+|\d+', response_text)
        if numbers:
            model_answer = numbers[-1].replace(",", "").replace("$", "")

    return thinking_text, response_text, model_answer

# Iterate through problems and modes
for mode_name, generation_params in {
    "thinking": GENERATION_CONFIG_THINKING_ON,
    "no_thinking": GENERATION_CONFIG_THINKING_OFF,
}.items():
    print(f"\n--- Running in {mode_name.upper()} mode ---")

    # Use tqdm for a progress bar over batches
    for i in tqdm(range(0, len(problems), BATCH_SIZE), desc=f"{mode_name} mode progress"):
        batch_problems = problems.select(range(i, min(i + BATCH_SIZE, len(problems))))
        batch_prompts = [
            prepare_prompt(tokenizer, p["question"], enable_thinking=(mode_name == "thinking"))
            for p in batch_problems
        ]

        # Tokenize batch prompts
        # Ensure attention_mask is generated for proper padding handling
        tokenized_inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True)
        input_ids = tokenized_inputs.input_ids.to(model.device)
        attention_mask = tokenized_inputs.attention_mask.to(model.device)

        # Generate responses
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=generation_params["max_new_tokens"],
                do_sample=True,
                temperature=generation_params["temperature"],
                top_p=generation_params["top_p"],
                top_k=generation_params["top_k"],
                pad_token_id=tokenizer.eos_token_id, # Qwen uses EOS token for padding
                return_dict_in_generate=True,
                output_scores=False, # No need for scores for this task
                stopping_criteria=None, # No specific stopping criteria beyond max_new_tokens
                eos_token_id=tokenizer.eos_token_id
            )

        for j, problem_data in enumerate(batch_problems):
            # Extract only the newly generated tokens (excluding prompt tokens)
            generated_tokens = outputs.sequences[j][len(input_ids[j]):].tolist()
            num_generated_tokens = len(generated_tokens)

            # Check if max_new_tokens was hit
            # This can be approximated by checking if the number of generated tokens is very close to max_new_tokens
            hit_max_tokens = num_generated_tokens >= generation_params["max_new_tokens"]

            thinking_text, response_text, model_answer = parse_model_output(generated_tokens, tokenizer)

            # Compare model's answer to gold answer (both converted to string for consistent comparison)
            correct = str(model_answer) == str(problem_data["gold_answer"])

            result = {
                "id": problem_data["id"],
                "question": problem_data["question"],
                "gold_answer": problem_data["gold_answer"],
                "mode": mode_name,
                "thinking": thinking_text,
                "response": response_text,
                "model_answer": model_answer,
                "correct": correct,
                "num_generated_tokens": num_generated_tokens,
                "hit_max_tokens": hit_max_tokens,
            }
            all_results.append(result)

            # Save after each problem within the batch
            # This ensures progress isn't lost if Colab disconnects
            with open(OUTPUT_FILE, 'a') as f:
                f.write(json.dumps(result) + '\n')

print("Generation and parsing complete. Results saved to JSONL file.")


--- Running in THINKING mode ---


thinking mode progress:   0%|          | 0/4 [00:00<?, ?it/s]


--- Running in NO_THINKING mode ---


no_thinking mode progress:   0%|          | 0/4 [00:00<?, ?it/s]

Generation and parsing complete. Results saved to JSONL file.


### 7. Summary and Analysis

In [14]:
import pandas as pd
import json
import os

# Reload results from the JSONL file to ensure all data is present
loaded_results = []
# Ensure the file exists before attempting to read it
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_results.append(json.loads(line))
else:
    print(f"Warning: Output file '{OUTPUT_FILE}' not found. No results to summarize.")

if loaded_results:
    df = pd.DataFrame(loaded_results)

    print("\n--- Summary Results ---")

    # Calculate accuracy per mode
    accuracy_per_mode = df.groupby('mode')['correct'].mean() * 100
    print("Accuracy per mode:")
    print(accuracy_per_mode)

    # Calculate average tokens per mode
    avg_tokens_per_mode = df.groupby('mode')['num_generated_tokens'].mean()
    print("\nAverage generated tokens per mode:")
    print(avg_tokens_per_mode)

    # Calculate how many outputs hit the max token limit
    hit_max_tokens_count = df.groupby('mode')['hit_max_tokens'].sum()
    print("\nNumber of outputs that hit max tokens limit per mode:")
    print(hit_max_tokens_count)

    print("\n--- Example Outputs ---")

    # Print 2 full example outputs (one per mode)
    # Example for 'thinking' mode
    if 'thinking' in df['mode'].values:
        print("\nExample for 'thinking' mode:")
        thinking_example = df[df['mode'] == 'thinking'].iloc[0]
        print(f"ID: {thinking_example['id']}")
        print(f"Question: {thinking_example['question']}")
        print(f"Gold Answer: {thinking_example['gold_answer']}")
        print(f"Mode: {thinking_example['mode']}")
        print(f"Thinking Process:\n{thinking_example['thinking']}")
        print(f"Response:\n{thinking_example['response']}")
        print(f"Model Answer: {thinking_example['model_answer']}")
        print(f"Correct: {thinking_example['correct']}")
        print(f"Generated Tokens: {thinking_example['num_generated_tokens']}")
        print(f"Hit Max Tokens: {thinking_example['hit_max_tokens']}")
    else:
        print("\nNo 'thinking' mode examples found.")

    # Example for 'no_thinking' mode
    if 'no_thinking' in df['mode'].values:
        print("\nExample for 'no_thinking' mode:")
        no_thinking_example = df[df['mode'] == 'no_thinking'].iloc[0]
        print(f"ID: {no_thinking_example['id']}")
        print(f"Question: {no_thinking_example['question']}")
        print(f"Gold Answer: {no_thinking_example['gold_answer']}")
        print(f"Mode: {no_thinking_example['mode']}")
        print(f"Response:\n{no_thinking_example['response']}")
        print(f"Model Answer: {no_thinking_example['model_answer']}")
        print(f"Correct: {no_thinking_example['correct']}")
        print(f"Generated Tokens: {no_thinking_example['num_generated_tokens']}")
        print(f"Hit Max Tokens: {no_thinking_example['hit_max_tokens']}")
    else:
        print("\nNo 'no_thinking' mode examples found.")

    print("\nAll tasks completed!")
else:
    print("No results were loaded for summary. Please ensure the generation step completed successfully.")


--- Summary Results ---
Accuracy per mode:
mode
no_thinking    50.0
thinking       70.0
Name: correct, dtype: float64

Average generated tokens per mode:
mode
no_thinking     587.25
thinking       2048.00
Name: num_generated_tokens, dtype: float64

Number of outputs that hit max tokens limit per mode:
mode
no_thinking     5
thinking       20
Name: hit_max_tokens, dtype: int64

--- Example Outputs ---

Example for 'thinking' mode:
ID: 0
Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Gold Answer: 18
Mode: thinking
Thinking Process:
<think>
Okay, let's see. Janet has ducks that lay eggs. The problem says she has 16 eggs per day. Then she eats three for breakfast every morning. Wait, but does that affect the number of eggs she sells? Hmm. Let me thi

### 3. Prompt Construction

In [9]:
# Helper function to create prompt messages
def create_messages(question, enable_thinking):
    user_message = (
        f"{question}\nSolve this step by step. End your response with 'Final answer: <number>'."
    )
    messages = [
        {"role": "user", "content": user_message}
    ]
    return messages

# Helper function to prepare the prompt for the model
def prepare_prompt(tokenizer, question, enable_thinking):
    messages = create_messages(question, enable_thinking)
    # The enable_thinking argument is passed directly to apply_chat_template
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=enable_thinking
    )
    return prompt

print("Prompt preparation functions defined.")

# Example of a thinking prompt
example_question = problems[0]["question"]
print("\n--- Example Prompt (Thinking ON) ---")
print(prepare_prompt(tokenizer, example_question, enable_thinking=True))

print("\n--- Example Prompt (Thinking OFF) ---")
print(prepare_prompt(tokenizer, example_question, enable_thinking=False))


Prompt preparation functions defined.

--- Example Prompt (Thinking ON) ---
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Solve this step by step. End your response with 'Final answer: <number>'.<|im_end|>
<|im_start|>assistant


--- Example Prompt (Thinking OFF) ---
<|im_start|>user
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Solve this step by step. End your response with 'Final answer: <number>'.<|im_end|>
<|im_start|>assistant
<think>

</think>






```
# This is formatted as code
```

### 4. Generation & 5. Parsing

In [10]:
# Generation parameters for thinking mode ON
GENERATION_CONFIG_THINKING_ON = {
    "temperature": 0.6,
    "top_p": 0.95,
    "top_k": 20,
    "max_new_tokens": 2048,
}

# Generation parameters for thinking mode OFF
GENERATION_CONFIG_THINKING_OFF = {
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "max_new_tokens": 1024,
}

BATCH_SIZE = 5

# List to store all results
all_results = []

# Function to parse model output
def parse_model_output(output, tokenizer):
    # Split the generated tokens at the '</think>' token (id 151668)
    # Qwen tokenizer has 151668 as the token ID for </think>
    thinking_token_id = 151668

    if thinking_token_id in output:
        thinking_tokens = output[:output.index(thinking_token_id)]
        response_tokens = output[output.index(thinking_token_id) + 1:]
        thinking_text = tokenizer.decode(thinking_tokens, skip_special_tokens=True)
        response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    else:
        thinking_text = ""
        response_text = tokenizer.decode(output, skip_special_tokens=True)

    # Extract the model's answer: the number after "Final answer:" or the last number
    model_answer = None
    final_answer_match = re.search(r"Final answer:\s*([\d,\.$]+)", response_text, re.IGNORECASE)
    if final_answer_match:
        model_answer = final_answer_match.group(1).replace(",", "").replace("$", "")
    else:
        # Fallback: if 'Final answer:' is not found, use the last number in the response
        numbers = re.findall(r'\d+\.\d+|\d+', response_text)
        if numbers:
            model_answer = numbers[-1].replace(",", "").replace("$", "")

    return thinking_text, response_text, model_answer


# Iterate through problems and modes
for mode_name, generation_params in {
    "thinking": GENERATION_CONFIG_THINKING_ON,
    "no_thinking": GENERATION_CONFIG_THINKING_OFF,
}.items():
    print(f"\n--- Running in {mode_name.upper()} mode ---")

    for i in tqdm(range(0, len(problems), BATCH_SIZE), desc=f"{mode_name} mode progress"):
        batch_problems = problems[i : i + BATCH_SIZE]
        batch_prompts = [
            prepare_prompt(tokenizer, p["question"], enable_thinking=(mode_name == "thinking"))
            for p in batch_problems
        ]

        # Tokenize batch prompts
        input_ids = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).input_ids
        input_ids = input_ids.to(model.device)

        # Generate responses
        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_new_tokens=generation_params["max_new_tokens"],
                do_sample=True,
                temperature=generation_params["temperature"],
                top_p=generation_params["top_p"],
                top_k=generation_params["top_k"],
                pad_token_id=tokenizer.eos_token_id, # Qwen uses EOS token for padding
                return_dict_in_generate=True, output_scores=True
            )

        for j, problem_data in enumerate(batch_problems):
            # Extract only the newly generated tokens
            generated_tokens = outputs.sequences[j][len(input_ids[j]):].tolist()
            num_generated_tokens = len(generated_tokens)

            # Check if max_new_tokens was hit
            hit_max_tokens = num_generated_tokens >= generation_params["max_new_tokens"]

            thinking_text, response_text, model_answer = parse_model_output(generated_tokens, tokenizer)

            correct = str(model_answer) == str(problem_data["gold_answer"])

            result = {
                "id": problem_data["id"],
                "question": problem_data["question"],
                "gold_answer": problem_data["gold_answer"],
                "mode": mode_name,
                "thinking": thinking_text,
                "response": response_text,
                "model_answer": model_answer,
                "correct": correct,
                "num_generated_tokens": num_generated_tokens,
                "hit_max_tokens": hit_max_tokens,
            }
            all_results.append(result)

            # Save after each batch
            with open(OUTPUT_FILE, 'a') as f:
                f.write(json.dumps(result) + '\n')

print("Generation and parsing complete. Results saved to JSONL file.")


--- Running in THINKING mode ---


thinking mode progress:   0%|          | 0/4 [00:00<?, ?it/s]

TypeError: string indices must be integers, not 'str'

### 7. Summary and Analysis

In [11]:
import pandas as pd
import json

# Reload results from the JSONL file to ensure all data is present
loaded_results = []
# Ensure the file exists before attempting to read it
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_results.append(json.loads(line))
else:
    print(f"Warning: Output file '{OUTPUT_FILE}' not found. No results to summarize.")

if loaded_results:
    df = pd.DataFrame(loaded_results)

    print("\n--- Summary Results ---")

    # Calculate accuracy per mode
    accuracy_per_mode = df.groupby('mode')['correct'].mean() * 100
    print("Accuracy per mode:")
    print(accuracy_per_mode)

    # Calculate average tokens per mode
    avg_tokens_per_mode = df.groupby('mode')['num_generated_tokens'].mean()
    print("\nAverage generated tokens per mode:")
    print(avg_tokens_per_mode)

    # Calculate how many outputs hit the max token limit
    hit_max_tokens_count = df.groupby('mode')['hit_max_tokens'].sum()
    print("\nNumber of outputs that hit max tokens limit per mode:")
    print(hit_max_tokens_count)

    print("\n--- Example Outputs ---")

    # Print 2 full example outputs (one per mode)
    # Example for 'thinking' mode
    if 'thinking' in df['mode'].values:
        print("\nExample for 'thinking' mode:")
        thinking_example = df[df['mode'] == 'thinking'].iloc[0]
        print(f"ID: {thinking_example['id']}")
        print(f"Question: {thinking_example['question']}")
        print(f"Gold Answer: {thinking_example['gold_answer']}")
        print(f"Mode: {thinking_example['mode']}")
        print(f"Thinking Process:\n{thinking_example['thinking']}")
        print(f"Response:\n{thinking_example['response']}")
        print(f"Model Answer: {thinking_example['model_answer']}")
        print(f"Correct: {thinking_example['correct']}")
        print(f"Generated Tokens: {thinking_example['num_generated_tokens']}")
        print(f"Hit Max Tokens: {thinking_example['hit_max_tokens']}")
    else:
        print("\nNo 'thinking' mode examples found.")

    # Example for 'no_thinking' mode
    if 'no_thinking' in df['mode'].values:
        print("\nExample for 'no_thinking' mode:")
        no_thinking_example = df[df['mode'] == 'no_thinking'].iloc[0]
        print(f"ID: {no_thinking_example['id']}")
        print(f"Question: {no_thinking_example['question']}")
        print(f"Gold Answer: {no_thinking_example['gold_answer']}")
        print(f"Mode: {no_thinking_example['mode']}")
        print(f"Response:\n{no_thinking_example['response']}")
        print(f"Model Answer: {no_thinking_example['model_answer']}")
        print(f"Correct: {no_thinking_example['correct']}")
        print(f"Generated Tokens: {no_thinking_example['num_generated_tokens']}")
        print(f"Hit Max Tokens: {no_thinking_example['hit_max_tokens']}")
    else:
        print("\nNo 'no_thinking' mode examples found.")

    print("\nAll tasks completed!")
else:
    print("No results were loaded for summary. Please ensure the generation step completed successfully.")

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/qwen3_1.7b_gsm8k.jsonl'

### 8. Refined Analysis of Results

In [15]:
import pandas as pd
import json
import os

# --- 1. Reload results to ensure freshness ---
loaded_results = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_results.append(json.loads(line))
else:
    print(f"Error: Output file '{OUTPUT_FILE}' not found. Cannot perform refined analysis.")
    exit()

if not loaded_results:
    print("No results found in the output file. Cannot perform refined analysis.")
    exit()

df_refined = pd.DataFrame(loaded_results)
print(f"Successfully reloaded {len(df_refined)} results for refined analysis.")

# --- 2. For each mode, count how many rows have an empty "response" field ---
empty_response_counts = df_refined[df_refined['response'] == ''].groupby('mode').size()
print("\n--- Empty 'response' field counts per mode ---")
if not empty_response_counts.empty:
    print(empty_response_counts)
else:
    print("No empty responses found in any mode.")

# --- 3. Fix num_generated_tokens and recompute hit_max_tokens ---
# We will re-tokenize the 'thinking' and 'response' text to get a count
# of 'real' tokens, excluding special tokens that were skipped during decoding.
# The 'max_new_tokens' limits were different for each mode.

def calculate_corrected_tokens(row):
    thinking_len = len(tokenizer.encode(row['thinking'], add_special_tokens=False))
    response_len = len(tokenizer.encode(row['response'], add_special_tokens=False))
    # For 'no_thinking' mode, the 'thinking' field is empty, so its token length will be 0.
    corrected_num_tokens = thinking_len + response_len
    return corrected_num_tokens

def calculate_corrected_hit_max_tokens(row):
    max_tokens_limit = (GENERATION_CONFIG_THINKING_ON['max_new_tokens'] if row['mode'] == 'thinking'
                        else GENERATION_CONFIG_THINKING_OFF['max_new_tokens'])
    # A response hits max tokens if its corrected token count is equal to the limit
    # AND it did not generate an explicit EOS token (which is difficult to ascertain
    # from decoded text alone without the raw token IDs). For simplicity, we'll assume
    # reaching the max_new_tokens limit implies hitting the limit.
    return row['corrected_num_generated_tokens'] >= max_tokens_limit # Using >= to be safe

df_refined['corrected_num_generated_tokens'] = df_refined.apply(calculate_corrected_tokens, axis=1)
df_refined['corrected_hit_max_tokens'] = df_refined.apply(calculate_corrected_hit_max_tokens, axis=1)

# --- 4. Print corrected average tokens and limit hits per mode ---
print("\n--- Corrected Average Generated Tokens per mode ---")
corrected_avg_tokens_per_mode = df_refined.groupby('mode')['corrected_num_generated_tokens'].mean()
print(corrected_avg_tokens_per_mode)

print("\n--- Corrected Number of Outputs that Hit Max Tokens Limit per mode ---")
corrected_hit_max_tokens_count = df_refined.groupby('mode')['corrected_hit_max_tokens'].sum()
print(corrected_hit_max_tokens_count)

# --- 5. Recompute accuracy using only rows where thinking mode actually finished ---
#    (non-empty response), and report how many rows that is.
filtered_df_for_accuracy = df_refined[df_refined['response'] != '']

print("\n--- Recomputed Accuracy per mode (only non-empty responses) ---")
if not filtered_df_for_accuracy.empty:
    recomputed_accuracy_per_mode = filtered_df_for_accuracy.groupby('mode')['correct'].mean() * 100
    print(recomputed_accuracy_per_mode)

    print("\n--- Number of rows used for recomputed accuracy per mode ---")
    rows_for_accuracy_count = filtered_df_for_accuracy.groupby('mode').size()
    print(rows_for_accuracy_count)
else:
    print("No non-empty responses found to recompute accuracy.")


Successfully reloaded 40 results for refined analysis.

--- Empty 'response' field counts per mode ---
No empty responses found in any mode.

--- Corrected Average Generated Tokens per mode ---
mode
no_thinking     339.00
thinking       1478.65
Name: corrected_num_generated_tokens, dtype: float64

--- Corrected Number of Outputs that Hit Max Tokens Limit per mode ---
mode
no_thinking    1
thinking       6
Name: corrected_hit_max_tokens, dtype: int64

--- Recomputed Accuracy per mode (only non-empty responses) ---
mode
no_thinking    50.0
thinking       70.0
Name: correct, dtype: float64

--- Number of rows used for recomputed accuracy per mode ---
mode
no_thinking    20
thinking       20
dtype: int64


### 9. Re-evaluation of Truncated 'Thinking' Mode Problems and Final Analysis

In [16]:
import pandas as pd
import json
import os
import re
from tqdm.notebook import tqdm
import torch # Assuming torch is still available from previous cells

# --- 0. Generation parameters for rerunning truncated thinking mode problems ---
GENERATION_CONFIG_TRUNCATED_RERUN = {
    "temperature": GENERATION_CONFIG_THINKING_ON["temperature"], # Same as original thinking mode
    "top_p": GENERATION_CONFIG_THINKING_ON["top_p"],      # Same as original thinking mode
    "top_k": GENERATION_CONFIG_THINKING_ON["top_k"],        # Same as original thinking mode
    "max_new_tokens": 8192, # Increased max_new_tokens
}

# --- 1. Reload results to identify truncated thinking-mode rows ---
loaded_results = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_results.append(json.loads(line))
else:
    print(f"Error: Output file '{OUTPUT_FILE}' not found. Cannot perform re-evaluation.")
    exit()

if not loaded_results:
    print("No results found in the output file. Cannot perform re-evaluation.")
    exit()

df_initial = pd.DataFrame(loaded_results)
print(f"Successfully loaded {len(df_initial)} initial results.")

# Identify truncated thinking-mode problems (where 'thinking' is empty and mode is 'thinking')
# This uses the definition of truncation from the initial run based on the parse_model_output function
truncated_thinking_problems_initial = df_initial[
    (df_initial['mode'] == 'thinking') & (df_initial['thinking'] == '')
]

print(f"Found {len(truncated_thinking_problems_initial)} initially truncated 'thinking' mode problems.")
if not truncated_thinking_problems_initial.empty:
    print("IDs of initially truncated problems:", truncated_thinking_problems_initial['id'].tolist())

# --- 2. Define the updated parsing function ---
def parse_model_output_v2(generated_tokens_list, tokenizer):
    thinking_token_id = 151668 # Token ID for </think>
    full_decoded_output = tokenizer.decode(generated_tokens_list, skip_special_tokens=True).strip()

    if thinking_token_id in generated_tokens_list:
        # Split tokens based on </think>
        split_index = generated_tokens_list.index(thinking_token_id)
        thinking_tokens = generated_tokens_list[:split_index]
        response_tokens = generated_tokens_list[split_index + 1:]

        thinking_text = tokenizer.decode(thinking_tokens, skip_special_tokens=True).strip()
        response_text = tokenizer.decode(response_tokens, skip_special_tokens=True).strip()
        is_truncated = False
    else:
        # If </think> token was not generated, the entire output is considered thinking
        thinking_text = full_decoded_output
        response_text = ""
        is_truncated = True

    model_answer = None
    # Only try to extract model_answer if there's a response (i.e., not truncated in the new sense)
    if not is_truncated and response_text: # Ensure response_text is not empty for extraction
        final_answer_match = re.search(r"Final answer:\s*([\d,\.$]+)", response_text, re.IGNORECASE)
        if final_answer_match:
            model_answer = final_answer_match.group(1).replace(",", "").replace("$", "")
        else:
            numbers = re.findall(r'\d+\.\d+|\d+', response_text)
            if numbers:
                model_answer = numbers[-1].replace(",", "").replace("$", "")

    return thinking_text, response_text, model_answer, is_truncated


# --- 3. Rerun ONLY the truncated thinking-mode problems ---
# We need the original 'problems' dataset which should be in the global scope from previous cell runs
# from datasets import Dataset
# problems_dataset = Dataset.from_pandas(df_initial.set_index('id')) # Assuming 'id' is unique and problem data consistent

rerun_results = []
if not truncated_thinking_problems_initial.empty:
    print("\n--- Rerunning initially truncated 'thinking' mode problems with max_new_tokens=8192 ---")
    rerun_problem_ids = truncated_thinking_problems_initial['id'].tolist()

    # Filter the original 'problems' dataset to get the actual problem objects for rerunning
    # The 'problems' variable is a Hugging Face Dataset from an earlier cell
    problems_to_rerun = problems.filter(lambda x: x['id'] in rerun_problem_ids)

    # Use a BATCH_SIZE for rerunning as well
    for i in tqdm(range(0, len(problems_to_rerun), BATCH_SIZE), desc="Rerunning truncated problems"):
        batch_problems_to_rerun = problems_to_rerun.select(range(i, min(i + BATCH_SIZE, len(problems_to_rerun))))
        batch_prompts_rerun = [
            prepare_prompt(tokenizer, p["question"], enable_thinking=True)
            for p in batch_problems_to_rerun
        ]

        tokenized_inputs_rerun = tokenizer(batch_prompts_rerun, return_tensors="pt", padding=True, truncation=True)
        input_ids_rerun = tokenized_inputs_rerun.input_ids.to(model.device)
        attention_mask_rerun = tokenized_inputs_rerun.attention_mask.to(model.device)

        with torch.no_grad():
            outputs_rerun = model.generate(
                input_ids_rerun,
                attention_mask=attention_mask_rerun,
                max_new_tokens=GENERATION_CONFIG_TRUNCATED_RERUN["max_new_tokens"],
                do_sample=True,
                temperature=GENERATION_CONFIG_TRUNCATED_RERUN["temperature"],
                top_p=GENERATION_CONFIG_TRUNCATED_RERUN["top_p"],
                top_k=GENERATION_CONFIG_TRUNCATED_RERUN["top_k"],
                pad_token_id=tokenizer.eos_token_id,
                return_dict_in_generate=True,
                output_scores=False,
                stopping_criteria=None,
                eos_token_id=tokenizer.eos_token_id
            )

        for j, problem_data_rerun in enumerate(batch_problems_to_rerun):
            generated_tokens_rerun = outputs_rerun.sequences[j][len(input_ids_rerun[j]):].tolist()

            # Use the updated parser
            thinking_text_rerun, response_text_rerun, model_answer_rerun, is_truncated_rerun = \
                parse_model_output_v2(generated_tokens_rerun, tokenizer)

            # Recalculate corrected tokens based on decoded text
            corrected_num_tokens_rerun = (
                len(tokenizer.encode(thinking_text_rerun, add_special_tokens=False)) +
                len(tokenizer.encode(response_text_rerun, add_special_tokens=False))
            )
            # Recalculate hit_max_tokens using the new limit and corrected token count
            corrected_hit_max_tokens_rerun = corrected_num_tokens_rerun >= GENERATION_CONFIG_TRUNCATED_RERUN["max_new_tokens"]

            # Compare model's answer to gold answer (only if not truncated)
            correct_rerun = False
            if not is_truncated_rerun and model_answer_rerun is not None: # Check for None model_answer
                correct_rerun = str(model_answer_rerun) == str(problem_data_rerun["gold_answer"])

            result_rerun = {
                "id": problem_data_rerun["id"],
                "question": problem_data_rerun["question"],
                "gold_answer": problem_data_rerun["gold_answer"],
                "mode": "thinking", # Still thinking mode
                "thinking": thinking_text_rerun,
                "response": response_text_rerun,
                "model_answer": model_answer_rerun,
                "correct": correct_rerun,
                "num_generated_tokens": len(generated_tokens_rerun), # Raw token count
                "hit_max_tokens": (len(generated_tokens_rerun) >= GENERATION_CONFIG_TRUNCATED_RERUN["max_new_tokens"]), # Raw hit max tokens
                "corrected_num_generated_tokens": corrected_num_tokens_rerun,
                "corrected_hit_max_tokens": corrected_hit_max_tokens_rerun,
                "is_truncated_v2": is_truncated_rerun, # New flag for the new truncation definition
            }
            rerun_results.append(result_rerun)

# --- 4. Update the JSONL file with new rows ---
# First, collect all rows that were NOT initially identified as truncated thinking-mode
non_truncated_initial_results = df_initial[
    ~((df_initial['mode'] == 'thinking') & (df_initial['thinking'] == ''))
].to_dict(orient='records')

# Combine non-truncated initial results with rerun results
final_results_list = non_truncated_initial_results + rerun_results

# Overwrite the OUTPUT_FILE with the combined results
print(f"\nOverwriting '{OUTPUT_FILE}' with updated results.")
with open(OUTPUT_FILE, 'w') as f: # Use 'w' to overwrite
    for result in final_results_list:
        f.write(json.dumps(result) + '\n')

print("Output file updated successfully.")

# --- 5. Reload all data for final summary and analysis ---
loaded_results_final = []
with open(OUTPUT_FILE, 'r') as f:
    for line in f:
        loaded_results_final.append(json.loads(line))
df_final = pd.DataFrame(loaded_results_final)

print("\n--- Final Refined Summary ---")

# Ensure 'is_truncated_v2' is present in all rows for consistent filtering
if 'is_truncated_v2' not in df_final.columns:
    # For 'no_thinking' and initial 'thinking' problems that weren't rerun,
    # they would have had a non-empty 'thinking' field if </think> was found.
    # So, we can infer is_truncated_v2=False if thinking is not empty, else True.
    # However, for non-thinking, it's always False (no </think> expected).
    df_final['is_truncated_v2'] = False # Default for non-rerun problems
    df_final.loc[(df_final['mode'] == 'thinking') & (df_final['response'] == ''), 'is_truncated_v2'] = True


# Separate problems based on the new truncation flag
truncated_thinking_problems_final = df_final[
    (df_final['mode'] == 'thinking') & (df_final['is_truncated_v2'] == True)
]

non_truncated_thinking_problems_final = df_final[
    (df_final['mode'] == 'thinking') & (df_final['is_truncated_v2'] == False)
]

no_thinking_problems_final = df_final[df_final['mode'] == 'no_thinking']


# Recompute accuracy per mode, counting truncated rows as a separate category
print(f"Number of truncated thinking-mode rows (where </think> was not found after re-run): {len(truncated_thinking_problems_final)}")
if not truncated_thinking_problems_final.empty:
    print("IDs of newly truncated problems:", truncated_thinking_problems_final['id'].tolist())


print("\n--- Recomputed Accuracy per mode (excluding truncated thinking-mode) ---")
if not non_truncated_thinking_problems_final.empty:
    accuracy_thinking_non_truncated = non_truncated_thinking_problems_final['correct'].mean() * 100
    print(f"Thinking mode (non-truncated): {accuracy_thinking_non_truncated:.2f}% ({len(non_truncated_thinking_problems_final)} problems)")
else:
    print("No non-truncated thinking mode problems to compute accuracy.")

if not no_thinking_problems_final.empty:
    accuracy_no_thinking = no_thinking_problems_final['correct'].mean() * 100
    print(f"No-thinking mode: {accuracy_no_thinking:.2f}% ({len(no_thinking_problems_final)} problems)")
else:
    print("No no-thinking mode problems to compute accuracy.")

print("\n--- Corrected Average Generated Tokens (using corrected token counts) ---")
# Use the corrected_num_generated_tokens
if not non_truncated_thinking_problems_final.empty:
    avg_tokens_thinking_non_truncated = non_truncated_thinking_problems_final['corrected_num_generated_tokens'].mean()
    print(f"Thinking mode (non-truncated): {avg_tokens_thinking_non_truncated:.2f}")
if not no_thinking_problems_final.empty:
    avg_tokens_no_thinking = no_thinking_problems_final['corrected_num_generated_tokens'].mean()
    print(f"No-thinking mode: {avg_tokens_no_thinking:.2f}")


print("\n--- Corrected Number of Outputs that Hit Max Tokens Limit (using corrected token counts) ---")
# Use the corrected_hit_max_tokens
if not non_truncated_thinking_problems_final.empty:
    hits_thinking_non_truncated = non_truncated_thinking_problems_final['corrected_hit_max_tokens'].sum()
    print(f"Thinking mode (non-truncated): {hits_thinking_non_truncated}")
if not no_thinking_problems_final.empty:
    hits_no_thinking = no_thinking_problems_final['corrected_hit_max_tokens'].sum()
    print(f"No-thinking mode: {hits_no_thinking}")


# Print the one no_thinking row that hit the limit
print("\n--- No-thinking mode problem that hit corrected max token limit ---")
no_thinking_hit_limit = df_final[
    (df_final['mode'] == 'no_thinking') & (df_final['corrected_hit_max_tokens'] == True)
]
if not no_thinking_hit_limit.empty:
    # Display the first one found if multiple exist
    print(no_thinking_hit_limit.iloc[0].to_dict())
else:
    print("No no-thinking mode problems hit the corrected max token limit.")


Successfully loaded 40 initial results.
Found 6 initially truncated 'thinking' mode problems.
IDs of initially truncated problems: ['2', '5', '7', '8', '12', '16']

--- Rerunning initially truncated 'thinking' mode problems with max_new_tokens=8192 ---


Filter:   0%|          | 0/20 [00:00<?, ? examples/s]

Rerunning truncated problems:   0%|          | 0/2 [00:00<?, ?it/s]


Overwriting 'outputs/qwen3_1.7b_gsm8k.jsonl' with updated results.
Output file updated successfully.

--- Final Refined Summary ---
Number of truncated thinking-mode rows (where </think> was not found after re-run): 0

--- Recomputed Accuracy per mode (excluding truncated thinking-mode) ---
Thinking mode (non-truncated): 83.33% (6 problems)
No-thinking mode: 50.00% (20 problems)

--- Corrected Average Generated Tokens (using corrected token counts) ---
Thinking mode (non-truncated): 2257.50
No-thinking mode: nan

--- Corrected Number of Outputs that Hit Max Tokens Limit (using corrected token counts) ---
Thinking mode (non-truncated): 0
No-thinking mode: 0

--- No-thinking mode problem that hit corrected max token limit ---
No no-thinking mode problems hit the corrected max token limit.


### 10. Inspection of `outputs/qwen3_1.7b_gsm8k.jsonl`

In [17]:
import pandas as pd
import json
import os

# --- Load the JSONL file ---
loaded_data_for_inspection = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_data_for_inspection.append(json.loads(line))
    df_inspect = pd.DataFrame(loaded_data_for_inspection)
    print(f"Successfully loaded {len(df_inspect)} rows from '{OUTPUT_FILE}'.")
else:
    print(f"Error: Output file '{OUTPUT_FILE}' not found for inspection.")
    exit()

# 1. Print the total number of rows, and rows per mode
print(f"\nTotal number of rows: {len(df_inspect)}")
print("Rows per mode:")
print(df_inspect['mode'].value_counts())

# 2. Print the IDs present for each mode
print("\nIDs per mode:")
for mode in df_inspect['mode'].unique():
    ids_in_mode = df_inspect[df_inspect['mode'] == mode]['id'].tolist()
    print(f"  {mode}: {ids_in_mode}")

# 3. Print which fields each row has, and flag rows missing 'corrected_num_generated_tokens' or 'is_truncated_v2'
print("\n--- Field Presence Check ---")
missing_corrected_tokens_ids = []
missing_truncated_flag_ids = []

for index, row in df_inspect.iterrows():
    if 'corrected_num_generated_tokens' not in row:
        missing_corrected_tokens_ids.append(row['id'])
    if 'is_truncated_v2' not in row:
        # Note: The user mentioned 'truncated', but the code uses 'is_truncated_v2'
        missing_truncated_flag_ids.append(row['id'])

if missing_corrected_tokens_ids:
    print(f"Rows missing 'corrected_num_generated_tokens': {len(missing_corrected_tokens_ids)} (IDs: {missing_corrected_tokens_ids})")
else:
    print("All rows have 'corrected_num_generated_tokens'.")

if missing_truncated_flag_ids:
    print(f"Rows missing 'is_truncated_v2': {len(missing_truncated_flag_ids)} (IDs: {missing_truncated_flag_ids})")
else:
    print("All rows have 'is_truncated_v2'.")

# 4. If any thinking-mode rows are missing, list the missing ids.
missing_thinking_mode_fields_ids = []
for index, row in df_inspect[df_inspect['mode'] == 'thinking'].iterrows():
    if 'corrected_num_generated_tokens' not in row or 'is_truncated_v2' not in row:
        missing_thinking_mode_fields_ids.append(row['id'])

if missing_thinking_mode_fields_ids:
    print(f"\nMissing fields in thinking-mode rows (IDs): {missing_thinking_mode_fields_ids}")
else:
    print("\nNo thinking-mode rows are missing the specified fields ('corrected_num_generated_tokens' or 'is_truncated_v2').")


Successfully loaded 40 rows from 'outputs/qwen3_1.7b_gsm8k.jsonl'.

Total number of rows: 40
Rows per mode:
mode
thinking       20
no_thinking    20
Name: count, dtype: int64

IDs per mode:
  thinking: ['0', '1', '3', '4', '6', '9', '10', '11', '13', '14', '15', '17', '18', '19', '2', '5', '7', '8', '12', '16']
  no_thinking: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19']

--- Field Presence Check ---
All rows have 'corrected_num_generated_tokens'.
All rows have 'is_truncated_v2'.

No thinking-mode rows are missing the specified fields ('corrected_num_generated_tokens' or 'is_truncated_v2').


### 11. Final Comprehensive Summary

In [19]:
import pandas as pd
import json
import os

# Assuming `tokenizer`, `GENERATION_CONFIG_THINKING_ON`, `GENERATION_CONFIG_THINKING_OFF` are available from previous cells.
# If not, they would need to be re-imported or re-defined here for the token correction functions.

# --- Reload all data for final summary and analysis from the potentially updated OUTPUT_FILE ---
loaded_results_final = []
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r') as f:
        for line in f:
            loaded_results_final.append(json.loads(line))
else:
    print(f"Error: Output file '{OUTPUT_FILE}' not found. Cannot perform final analysis.")
    exit()

df_final = pd.DataFrame(loaded_results_final)

print("\n--- df_final Info (before re-calculation) ---")
df_final.info()
print("\n--- df_final Head (before re-calculation) ---")
display(df_final.head())

# --- Re-define token correction functions to ensure availability ---
def calculate_corrected_tokens(row):
    # Ensure 'thinking' and 'response' fields are treated as strings to avoid errors with non-string values
    thinking_text = str(row['thinking'])
    response_text = str(row['response'])
    thinking_len = len(tokenizer.encode(thinking_text, add_special_tokens=False))
    response_len = len(tokenizer.encode(response_text, add_special_tokens=False))
    corrected_num_tokens = thinking_len + response_len
    return corrected_num_tokens

def calculate_corrected_hit_max_tokens(row):
    max_tokens_limit = (GENERATION_CONFIG_THINKING_ON['max_new_tokens'] if row['mode'] == 'thinking'
                        else GENERATION_CONFIG_THINKING_OFF['max_new_tokens'])
    # A response hits max tokens if its corrected token count is equal to or greater than the limit
    return row['corrected_num_generated_tokens'] >= max_tokens_limit

# --- Ensure 'corrected_num_generated_tokens', 'corrected_hit_max_tokens', and 'is_truncated_v2' are populated for ALL rows ---

# Calculate corrected_num_generated_tokens for all rows if not already present/null
# Use .loc to avoid SettingWithCopyWarning
df_final.loc[df_final['corrected_num_generated_tokens'].isnull(), 'corrected_num_generated_tokens'] = \
    df_final[df_final['corrected_num_generated_tokens'].isnull()].apply(calculate_corrected_tokens, axis=1)

# Ensure corrected_num_generated_tokens is numeric (float or int)
df_final['corrected_num_generated_tokens'] = df_final['corrected_num_generated_tokens'].astype(float)

# Calculate corrected_hit_max_tokens for all rows based on the corrected token count
df_final.loc[:, 'corrected_hit_max_tokens'] = df_final.apply(calculate_corrected_hit_max_tokens, axis=1)

# Ensure 'is_truncated_v2' is present and correctly set for all rows
# For no_thinking mode, it's always False as there's no </think> token expected.
# For thinking mode problems not re-run, infer based on original logic (empty thinking text = truncated)
# This needs to be carefully applied. The re-run thinking mode problems already have it.
# For initial thinking mode results (those not re-run), if 'thinking' is empty, it was considered truncated initially.
# For no_thinking, it should always be False.

# Initialize 'is_truncated_v2' to False for all rows that don't have it (e.g., no_thinking or non-rerun thinking)
if 'is_truncated_v2' not in df_final.columns:
    df_final['is_truncated_v2'] = False

# For rows that still have null in 'is_truncated_v2' (which implies they were original thinking mode and not re-run)
# or for 'no_thinking' rows which would be NaN if a subset was processed, set explicitly.
# More robust way: only 'thinking' mode can be truncated. 'no_thinking' is never truncated in this context.
# For thinking mode, if response is empty after parsing (and not part of rerun that sets is_truncated_v2), it's truncated.

df_final.loc[df_final['mode'] == 'no_thinking', 'is_truncated_v2'] = False

# For thinking mode problems that were not part of the re-run (i.e. 'is_truncated_v2' might be null)
# and whose `thinking` field is empty, set `is_truncated_v2` to True.
# The `apply` function with `parse_model_output_v2` for the initial data would be more robust,
# but for simplicity, we'll use the 'thinking' column itself as an indicator for pre-rerun data.
# However, the current state of `df_final` implies `is_truncated_v2` is only non-null for the 6 re-run thinking problems.
# So, the NaN values need to be filled. Let's assume for any non-rerun thinking problem, if its original 'thinking' field was empty, it implies truncation.
# A simpler approach: for any 'thinking' mode entry, if the response is empty, consider it truncated for summary purposes, otherwise not.
# This ensures consistency.

df_final.loc[(df_final['mode'] == 'thinking') & (df_final['response'] == ''), 'is_truncated_v2'] = True
# For all other thinking mode problems where response is not empty, it's not truncated.
df_final.loc[(df_final['mode'] == 'thinking') & (df_final['response'] != ''), 'is_truncated_v2'] = False


print("\n--- df_final Info (after re-calculation) ---")
df_final.info()
print("\n--- df_final Head (after re-calculation) ---")
display(df_final.head())


print("\n--- Final Summary per Mode ---")

summary_data = []

for mode in df_final['mode'].unique():
    mode_df = df_final[df_final['mode'] == mode].copy()

    # Total problems in this mode
    total_problems = len(mode_df)

    # Truncated problems (where is_truncated_v2 is True)
    # Note: is_truncated_v2 is only relevant for 'thinking' mode. For 'no_thinking', it should always be False.
    truncated_problems = mode_df['is_truncated_v2'].sum()

    # Non-truncated problems for accuracy calculation
    non_truncated_df = mode_df[mode_df['is_truncated_v2'] == False]
    num_non_truncated = len(non_truncated_df)

    # Accuracy (excluding truncated rows)
    accuracy = (non_truncated_df['correct'].mean() * 100) if not non_truncated_df.empty else None

    # Average generated tokens (using corrected_num_generated_tokens)
    avg_tokens = non_truncated_df['corrected_num_generated_tokens'].mean() if not non_truncated_df.empty else None

    # Max generated tokens (using corrected_num_generated_tokens)
    max_tokens = non_truncated_df['corrected_num_generated_tokens'].max() if not non_truncated_df.empty else None

    # Number of outputs that hit max tokens limit (using corrected_hit_max_tokens)
    hits_max_tokens = non_truncated_df['corrected_hit_max_tokens'].sum() if not non_truncated_df.empty else 0

    summary_data.append({
        'Mode': mode,
        'Total Problems': total_problems,
        'Truncated Problems': int(truncated_problems), # Convert to int for display
        'Non-Truncated for Accuracy': num_non_truncated,
        'Accuracy (%)': f"{accuracy:.2f}" if accuracy is not None else "N/A",
        'Avg Corrected Tokens': f"{avg_tokens:.2f}" if avg_tokens is not None else "N/A",
        'Max Corrected Tokens': int(max_tokens) if max_tokens is not None else "N/A",
        'Hits Max Tokens Limit': int(hits_max_tokens) # Convert to int for display
    })

summary_df = pd.DataFrame(summary_data).set_index('Mode')
display(summary_df)


print("\n--- Comparative Performance: Thinking vs. No-Thinking ---")

# Merge dataframes for comparison
comparison_df = pd.merge(
    df_final[df_final['mode'] == 'thinking'][['id', 'correct', 'is_truncated_v2']],
    df_final[df_final['mode'] == 'no_thinking'][['id', 'correct', 'is_truncated_v2']], # Also include is_truncated_v2 for no_thinking for completeness
    on='id',
    suffixes=('_thinking', '_no_thinking')
)

# Scenario 1: Thinking mode correct, No-thinking mode wrong
# Exclude cases where thinking mode itself was truncated
thinking_better_ids = comparison_df[
    (comparison_df['correct_thinking'] == True) &
    (comparison_df['is_truncated_v2_thinking'] == False) &
    (comparison_df['correct_no_thinking'] == False)
]['id'].tolist()

print(f"Problems where 'thinking' was correct and 'no_thinking' was wrong: {thinking_better_ids}")

# Scenario 2: No-thinking mode correct, Thinking mode wrong
# Exclude cases where thinking mode itself was truncated
no_thinking_better_ids = comparison_df[
    (comparison_df['correct_no_thinking'] == True) &
    (comparison_df['is_truncated_v2_no_thinking'] == False) &
    (comparison_df['correct_thinking'] == False)
]['id'].tolist()

print(f"Problems where 'no_thinking' was correct and 'thinking' was wrong: {no_thinking_better_ids}")



--- df_final Info (before re-calculation) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              40 non-null     object 
 1   question                        40 non-null     object 
 2   gold_answer                     40 non-null     object 
 3   mode                            40 non-null     object 
 4   thinking                        40 non-null     object 
 5   response                        40 non-null     object 
 6   model_answer                    39 non-null     object 
 7   correct                         40 non-null     bool   
 8   num_generated_tokens            40 non-null     int64  
 9   hit_max_tokens                  40 non-null     bool   
 10  corrected_num_generated_tokens  6 non-null      float64
 11  corrected_hit_max_tokens        6 non-null      obje

,id,question,gold_answer,mode,thinking,response,model_answer,correct,num_generated_tokens,hit_max_tokens,corrected_num_generated_tokens,corrected_hit_max_tokens,is_truncated_v2
0,0,Janet’s ducks lay 16 eggs per day. She eats th...,18,thinking,"<think>\nOkay, let's see. Janet has ducks that...",To determine how much Janet makes at the farme...,18,True,2048,True,NaN,NaN,NaN
1,1,A robe takes 2 bolts of blue fiber and half th...,3,thinking,"<think>\nOkay, let's see. The problem says a r...",To determine the total number of bolts require...,3,True,2048,True,NaN,NaN,NaN
2,3,James decides to run 3 sprints 3 times a week....,540,thinking,"<think>\nOkay, let's see. James runs 3 sprints...","James runs 3 sprints 3 times a week, meaning h...",540,True,2048,True,NaN,NaN,NaN
3,4,"Every day, Wendi feeds each of her chickens th...",20,thinking,"<think>\nOkay, let's see. Wendi feeds her chic...",To determine how many cups of feed Wendi needs...,20,True,2048,True,NaN,NaN,NaN
4,6,Toulouse has twice as many sheep as Charleston...,260,thinking,"<think>\nOkay, let's try to figure out this sh...","To solve the problem, we start by defining the...",260,True,2048,True,NaN,NaN,NaN



--- df_final Info (after re-calculation) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 13 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              40 non-null     object 
 1   question                        40 non-null     object 
 2   gold_answer                     40 non-null     object 
 3   mode                            40 non-null     object 
 4   thinking                        40 non-null     object 
 5   response                        40 non-null     object 
 6   model_answer                    39 non-null     object 
 7   correct                         40 non-null     bool   
 8   num_generated_tokens            40 non-null     int64  
 9   hit_max_tokens                  40 non-null     bool   
 10  corrected_num_generated_tokens  40 non-null     float64
 11  corrected_hit_max_tokens        40 non-null     objec

,id,question,gold_answer,mode,thinking,response,model_answer,correct,num_generated_tokens,hit_max_tokens,corrected_num_generated_tokens,corrected_hit_max_tokens,is_truncated_v2
0,0,Janet’s ducks lay 16 eggs per day. She eats th...,18,thinking,"<think>\nOkay, let's see. Janet has ducks that...",To determine how much Janet makes at the farme...,18,True,2048,True,1558.0,False,False
1,1,A robe takes 2 bolts of blue fiber and half th...,3,thinking,"<think>\nOkay, let's see. The problem says a r...",To determine the total number of bolts require...,3,True,2048,True,842.0,False,False
2,3,James decides to run 3 sprints 3 times a week....,540,thinking,"<think>\nOkay, let's see. James runs 3 sprints...","James runs 3 sprints 3 times a week, meaning h...",540,True,2048,True,737.0,False,False
3,4,"Every day, Wendi feeds each of her chickens th...",20,thinking,"<think>\nOkay, let's see. Wendi feeds her chic...",To determine how many cups of feed Wendi needs...,20,True,2048,True,1701.0,False,False
4,6,Toulouse has twice as many sheep as Charleston...,260,thinking,"<think>\nOkay, let's try to figure out this sh...","To solve the problem, we start by defining the...",260,True,2048,True,847.0,False,False



--- Final Summary per Mode ---


,Total Problems,Truncated Problems,Non-Truncated for Accuracy,Accuracy (%),Avg Corrected Tokens,Max Corrected Tokens,Hits Max Tokens Limit
Mode,,,,,,,
thinking,20,0,20,85.00,1541.50,5364,2
no_thinking,20,0,20,50.00,339.00,1024,1



--- Comparative Performance: Thinking vs. No-Thinking ---
Problems where 'thinking' was correct and 'no_thinking' was wrong: ['3', '4', '6', '13', '14', '19', '8', '12', '16']
Problems where 'no_thinking' was correct and 'thinking' was wrong: ['11', '17']
